# Thesis 最终版：复现、结果核对与改进分析

对照 `<THESIS_FINAL_PDF>`（77 页）。前半部分保留早期四模型资产检查；它们不是最终论文表 4.11 的基准。第 7 节起对照最终论文重新计算 EC 层级、宇宙映射、GO 富集及增强收益，并严格区分论文报告值和本次计算值。原文件仅移动，未改动内容。新结果写入独立运行目录。


In [ ]:
from pathlib import Path
import os, json, csv, re, ast, hashlib, sys, datetime, importlib.metadata
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import torch
import h5py
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(os.environ.get('THESIS_ROOT', Path.cwd())).resolve()
assert (ROOT / '01_code/Code').is_dir(), '请从整理后的 Thesis 根目录启动 notebook，或设置 THESIS_ROOT。'
CODE = ROOT / '01_code/Code'
DATA = ROOT / '02_data/data'
CLEAN = ROOT / '03_models_and_tools/tttianhao-CLEAN-0cf2cac'
RUN = ROOT / '04_results' / ('reproduction_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
RUN.mkdir(parents=True, exist_ok=False)
np.random.seed(1234); torch.manual_seed(1234); torch.set_num_threads(2)
versions = {k: importlib.metadata.version(k) for k in ['numpy','pandas','torch','h5py','scikit-learn','matplotlib']}
(RUN / 'environment.json').write_text(json.dumps({'python':sys.version,'packages':versions}, indent=2))
print('输出目录:', RUN)
display(versions)

def availability(p):
    p = Path(p)
    # 先检查祖先的 dataless 标记，避免列举云端目录导致阻塞。
    for q in reversed([p, *p.parents]):
        try:
            if getattr(q.lstat(), 'st_flags', 0) & 0x40000000:
                return 'cloud_only: ' + str(q.relative_to(ROOT) if q.is_relative_to(ROOT) else q)
        except FileNotFoundError:
            return 'missing: ' + str(q)
    return 'local'

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

## 1. 读取原始数据：保留历史口径，同时暴露表头问题
CLEAN 原 `get_ec_id_dict` 无条件跳过首行。本 notebook 的 `legacy=True` 精确保留这一行为；`legacy=False` 仅在真实表头存在时跳过。先看两个口径差异，再决定后续科学分析使用哪一个。**复现旧结果默认使用 legacy=True，不改原 CSV。**

In [ ]:
def read_labels(path, legacy=True):
    with Path(path).open() as f:
        rows = list(csv.reader(f, delimiter='\t'))
    assert rows and len(rows[0]) >= 2, path
    has_header = rows[0][0].strip().lower() in {'entry','id','protein_id','uniprot_id'}
    start = 1 if legacy or has_header else 0
    id_ec = {}; ec_id = defaultdict(set)
    for row in rows[start:]:
        if len(row) < 2: continue
        labels = row[1].split(';')
        id_ec[row[0]] = labels
        for ec in labels: ec_id[ec].add(row[0])
    return id_ec, dict(ec_id), has_header

counts=[]
for name in ['split10_re2','new2','split50','new','new_test','split100','split_con_augu_50']:
    p=CLEAN/'data'/f'{name}.csv'
    if availability(p)!='local':continue
    old, ecs, header=read_labels(p,True)
    corrected,_,_=read_labels(p,False)
    counts.append(dict(dataset=name,legacy_samples=len(old),header_aware_samples=len(corrected),legacy_ECs=len(ecs),has_header=header,sha256=sha256(p)))
counts=pd.DataFrame(counts)
display(counts.drop(columns='sha256'))
counts.to_csv(RUN/'dataset_counts.csv',index=False)
train_labels, train_ec, _ = read_labels(CLEAN/'data/split10_re2.csv',True)
test_labels, _, _ = read_labels(CLEAN/'data/new2.csv',True)
print('训练/测试 ID 交集:',len(set(train_labels)&set(test_labels)))
print('按 EC 重复展开的训练样本数（多标签蛋白会重复）:',sum(map(len,train_ec.values())))
# ID 不重叠不代表同源性隔离；本检查不能排除序列相似性泄漏。

## 2. 已保存的结构 / 功能标签资产
这部分从 HDF5 实际读取元信息，用来确认数据能否打开、维度和键是否存在；不是重新生成上游聚类，也不是 GO 富集结果。缺失的上游输入明确列出。

In [ ]:
assets=[]
for name in ['Fun_fold.h5','UniFun.h5','sepFun.h5','local2Universe.h5','Disen_trainData_Labels_int.h5','Disen_testData_Labels_int.h5']:
    p=DATA/name; status=availability(p)
    row={'file':name,'status':status}
    if status=='local':
        try:
            with h5py.File(p,'r') as f:
                key=next(iter(f),None)
                row.update(keys=len(f),example_key=key,example_shape=str(f[key].shape) if key else '')
        except Exception as e:row['status']=f'error: {type(e).__name__}: {e}'
    assets.append(row)
display(pd.DataFrame(assets))
upstream=pd.DataFrame([{'path':str(p.relative_to(ROOT)),'status':availability(p)} for p in [DATA/'all',DATA/'Embedding',DATA/'alignment_db',ROOT/'03_models_and_tools/sep_clr-master/Protein_Model']])
display(upstream)
upstream.to_csv(RUN/'upstream_status.csv',index=False)

## 2b. 实际重算：结构簇的 EC 一致性
从原 `Fun_fold.h5` 的蛋白→簇映射恢复簇成员，使用 `split100.csv` 的 EC 标签，重算 `1 Function Enrichment Analysis.ipynb` cell 17 / 25 的一致性分组和堆叠图。

这属于**从已保存中间数据重算**，不是从原始结构重跑 Foldseek。原计算比较 EC 列表（顺序敏感）；这里同时报告列表一致与集合一致。无法证明这份 H5 和历史图使用了完全相同的筛选快照，因此不宣称与历史图逐点相等。缺标签的簇不进入比例分母，并报告排除数量；不把“缺标签”当作“不一致”。

In [ ]:
mapping_path=DATA/'Fun_fold.h5'
cluster_summary=None
if availability(mapping_path)=='local' and availability(CLEAN/'data/split100.csv')=='local':
    all_labels,_,_=read_labels(CLEAN/'data/split100.csv',True)
    clusters=defaultdict(list)
    with h5py.File(mapping_path,'r') as f:
        for protein_id in f:
            v=f[protein_id][()]
            cluster=v.decode() if isinstance(v,bytes) else str(v)
            clusters[cluster].append(protein_id)
    rows=[]
    for cluster,members in clusters.items():
        labels=[all_labels[k] for k in members if k in all_labels]
        complete=len(labels)==len(members)
        rows.append(dict(cluster=cluster,size=len(members),labeled_members=len(labels),complete_labels=complete,
          list_consistent=all(x==labels[0] for x in labels) if complete and labels else None,
          set_consistent=all(set(x)==set(labels[0]) for x in labels) if complete and labels else None))
    cluster_summary=pd.DataFrame(rows)
    cluster_summary.to_csv(RUN/'cluster_ec_consistency.csv',index=False)
    print('保存映射中的成员数:',sum(map(len,clusters.values())),'簇数:',len(clusters))
    print('缺标签而排除的簇数:',int((~cluster_summary.complete_labels).sum()))
    display(cluster_summary.groupby(['complete_labels','list_consistent'],dropna=False).size().rename('clusters'))
    segments=[(1,2),(2,4),(4,10),(10,50),(50,100),(100,300),(300,600),(600,900)]
    bins=[]
    for a,b in segments:
        part=cluster_summary[(cluster_summary['size']>=a)&(cluster_summary['size']<b)&cluster_summary.complete_labels]
        yes=int(part.list_consistent.eq(True).sum());total=len(part)
        bins.append(dict(interval=f'[{a},{b})',consistent=yes,inconsistent=total-yes,total=total,consistent_pct=100*yes/total if total else np.nan))
    bins=pd.DataFrame(bins);display(bins);bins.to_csv(RUN/'cluster_size_bins.csv',index=False)
    print('原图范围外（size >=900）的簇:',int(cluster_summary['size'].ge(900).sum()))
    fig,ax=plt.subplots(figsize=(9,4));ax.bar(bins.interval,bins.consistent_pct,label='EC consistent',color='gray')
    ax.bar(bins.interval,100-bins.consistent_pct,bottom=bins.consistent_pct,label='EC inconsistent',color='salmon')
    ax.set_ylabel('Percent of fully labeled clusters');ax.set_xlabel('Cluster size');ax.legend();fig.tight_layout()
    fig.savefig(RUN/'ec_consistency_recomputed.png',dpi=160);plt.show()
else:
    print('BLOCKED: membership H5 or split100 labels unavailable.')


## 3. 从原 notebook 抽取历史对照
保留 notebook、cell index（从 0 开始）、源代码和保存输出。所有这些记录的状态都是 `historical_only`；各行样本数不同，不能把所有 F1 放在同一排行榜。

In [ ]:
historical=[]
for p in sorted(CODE.glob('*.ipynb')):
    d=json.loads(p.read_text())
    for i,c in enumerate(d.get('cells',[])):
        source=''.join(c.get('source',[]))
        for o in c.get('outputs',[]):
            t=o.get('text',o.get('data',{}).get('text/plain',[]));t=''.join(t) if isinstance(t,list) else t
            if '>>> precision:' not in t:continue
            model=re.search(r'model_name\s*=\s*["\']([^"\']+)',source)
            row={'notebook':p.name,'cell':i,'model':model.group(1) if model else '', 'status':'historical_only','source':source,'stored_output':t}
            for key,pattern in {'samples':r'total samples:\s*(\d+)','ec_count':r'total ec:\s*(\d+)','precision':r'precision:\s*([\d.]+)','recall':r'recall:\s*([\d.]+)','f1':r'F1:\s*([\d.]+)','auc':r'AUC:\s*([\d.]+)','accuracy':r'ACC:\s*([\d.]+)'}.items():
                match=re.search(pattern,t,re.I)
                row[key]=float(match.group(1)) if match else None
            historical.append(row)
history=pd.DataFrame(historical)
display(history.drop(columns=['source','stored_output']))
history.to_json(RUN/'historical_evidence.json',orient='records',force_ascii=False,indent=2)

## 4. 四个模型使用一套推断逻辑
依据原 `evaluate_models.ipynb` cell 9 / 13 / 15 / 17 配置输入。旧 notebook 中模型定义从 1024 输入变为导入类，同时 checkpoint 显示 1280 输入，存在交互状态依赖；这里以 checkpoint 权重形状构建完全匹配的 MLP，并严格核对 embedding 维度。

计算过程：embedding → 已训练 MLP → 每个 EC 的训练表示均值 → 测试表示到 EC 中心的欧氏距离 → 原 maximum-separation 规则 → 原多标签指标。只做推断复现，不从头训练。

In [ ]:
configs = [
 dict(model='split10_triplet',kind='esm',train='esm_data',test='esm_data'),
 dict(model='split10pro_triplet',kind='h5',train='prost5/some_embeddings.h5',test='prost5/New_embeddings.h5'),
 dict(model='split10_sub3Di_triplet',kind='h5',train='sub3Di/sub3Di.h5',test='sub3Di/new_sub3Di.h5'),
 dict(model='split10proSEQ_triplet',kind='h5',train='prost5/split10_seq.h5',test='prost5/new_seq.h5'),
]
class Projection(torch.nn.Module):
    def __init__(self,state):
        super().__init__()
        for i in (1,2,3):
            w=state[f'fc{i}.weight'];setattr(self,f'fc{i}',torch.nn.Linear(w.shape[1],w.shape[0]))
        self.ln1=torch.nn.LayerNorm(state['fc1.weight'].shape[0])
        self.ln2=torch.nn.LayerNorm(state['fc2.weight'].shape[0])
        self.dropout=torch.nn.Dropout(.1)
        self.load_state_dict(state,strict=True)
    def forward(self,x):
        x=torch.relu(self.dropout(self.ln1(self.fc1(x))))
        x=torch.relu(self.dropout(self.ln2(self.fc2(x))))
        return self.fc3(x)

# 从用户自己的 CLEAN 源文件只抽取四个已检查的纯函数，不 import 整个项目或运行顶层代码。
eval_path=CLEAN/'src/CLEAN/evaluate.py'
wanted={'maximum_separation','get_ec_pos_dict','get_eval_metrics'}
tree=ast.parse(eval_path.read_text())
functions=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in wanted]
assert {n.name for n in functions}==wanted
exec(compile(ast.Module(body=functions,type_ignores=[]),str(eval_path),'exec'),globals())
print('评估实现 SHA256:',sha256(eval_path))

checkpoint_rows=[]
for cfg in configs:
    p=CLEAN/'data/model'/f"{cfg['model']}.pth"
    row={'model':cfg['model'],'checkpoint_status':availability(p),'train_status':availability(CLEAN/'data'/cfg['train']),'test_status':availability(CLEAN/'data'/cfg['test'])}
    if row['checkpoint_status']=='local':
        state=torch.load(p,map_location='cpu',weights_only=True)
        model=Projection(state).eval()
        row.update(input_dim=state['fc1.weight'].shape[1],output_dim=state['fc3.weight'].shape[0],parameters=sum(x.numel() for x in model.parameters()),checkpoint_sha256=sha256(p))
    checkpoint_rows.append(row)
checkpoint_status=pd.DataFrame(checkpoint_rows)
display(checkpoint_status.drop(columns='checkpoint_sha256',errors='ignore'))
checkpoint_status.to_csv(RUN/'checkpoint_status.csv',index=False)

In [ ]:
def load_vectors(path, ids, kind):
    vectors=[]
    if kind=='h5':
        with h5py.File(path,'r') as f:
            missing=[k for k in ids if k not in f]
            if missing:raise ValueError(f'{len(missing)} missing embedding IDs; first: {missing[:5]}')
            for k in ids:
                a=np.asarray(f[k][()]).squeeze()
                if a.ndim!=1:raise ValueError(f'{k}: expected per-protein vector, got {a.shape}; no silent pooling')
                vectors.append(torch.as_tensor(a,dtype=torch.float32))
    else:
        for k in ids:
            p=path/f'{k}.pt'
            if availability(p)!='local':raise FileNotFoundError(str(p))
            a=torch.load(p,map_location='cpu',weights_only=True)
            if isinstance(a,dict):a=a['mean_representations'][33]
            a=a.squeeze().float()
            if a.ndim!=1:raise ValueError(f'{k}: expected vector, got {a.shape}')
            vectors.append(a)
    return torch.stack(vectors)

@torch.inference_mode()
def project_batches(model,vectors,batch_size=512):
    if vectors.shape[1]!=model.fc1.in_features:raise ValueError(f'Input dimension {vectors.shape[1]} != checkpoint {model.fc1.in_features}')
    return torch.cat([model(vectors[i:i+batch_size]) for i in range(0,len(vectors),batch_size)])

@torch.inference_mode()
def run_model(cfg):
    global MultiLabelBinarizer, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
    from sklearn.preprocessing import MultiLabelBinarizer
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
    checkpoint=CLEAN/'data/model'/f"{cfg['model']}.pth"
    state=torch.load(checkpoint,map_location='cpu',weights_only=True)
    model=Projection(state).eval()
    train_ids=list(train_labels);test_ids=list(test_labels)
    train_vectors=load_vectors(CLEAN/'data'/cfg['train'],train_ids,cfg['kind'])
    test_vectors=load_vectors(CLEAN/'data'/cfg['test'],test_ids,cfg['kind'])
    ztrain=project_batches(model,train_vectors);ztest=project_batches(model,test_vectors)
    id_to_idx={k:i for i,k in enumerate(train_ids)}
    ecs=list(train_ec)
    # 每个 EC 中每条蛋白只取一次；按固定顺序求均值，减少 set 顺序引起的浮点差异。
    centers=torch.stack([ztrain[[id_to_idx[k] for k in sorted(train_ec[ec])]].mean(0) for ec in ecs])
    distances=torch.stack([(z-centers).norm(dim=1) for z in ztest]).numpy()
    pred=[];probs=[];records=[]
    for protein_id,row in zip(test_ids,distances):
        nearest=pd.Series(row,index=ecs).nsmallest(10)
        cut=maximum_separation(nearest.tolist(),True,False)
        chosen=list(nearest.index[:cut+1])
        # 原实现写 CSV 时距离保留四位小数，再读入计算概率。
        d=torch.tensor([float(f'{v:.4f}') for v in nearest.iloc[:cut+1]],dtype=torch.float32)
        prob=(1-torch.exp(-1/d))/(1+torch.exp(-1/d));prob=prob/prob.sum()
        pred.append(chosen);probs.append(prob)
        records.append({'id':protein_id,'predicted_ECs':chosen,'distances_rounded':d.tolist()})
    truth=[test_labels[k] for k in test_ids];all_labels=set(ec for labs in truth for ec in labs)
    pre,rec,f1,auc,acc=get_eval_metrics(pred,probs,truth,all_labels)
    (RUN/f"{cfg['model']}_predictions.json").write_text(json.dumps(records,indent=2))
    return dict(model=cfg['model'],status='rerun',samples=len(truth),ec_count=len(all_labels),precision=pre,recall=rec,f1=f1,auc=auc,accuracy=acc)

## 5. 运行与缺失报告
缺少输入的模型明确跳过。没有以历史数字填充本次计算列。已恢复输入后重新 Run All 即可尝试推断；若 shape、ID 或指标不匹配，保留失败原因而不偷偷更换模型。

In [ ]:
reruns=[]
for cfg in configs:
    required=[CLEAN/'data/model'/f"{cfg['model']}.pth",CLEAN/'data'/cfg['train'],CLEAN/'data'/cfg['test']]
    blockers=[{'path':str(p.relative_to(ROOT)),'reason':availability(p)} for p in required if availability(p)!='local']
    if blockers:
        reruns.append({'model':cfg['model'],'status':'blocked','reason':json.dumps(blockers,ensure_ascii=False)})
        continue
    try:reruns.append(run_model(cfg))
    except Exception as e:reruns.append({'model':cfg['model'],'status':'failed','reason':f'{type(e).__name__}: {e}'})
reruns=pd.DataFrame(reruns)
display(reruns)
reruns.to_csv(RUN/'rerun_status.csv',index=False)
reference=history[history.notebook.eq('evaluate_models.ipynb')][['model','f1','auc']].rename(columns={'f1':'historical_f1','auc':'historical_auc'})
comparison=reference.merge(reruns,on='model',how='left')
if 'f1' in comparison:
    comparison['delta_f1']=comparison['f1']-comparison['historical_f1']
    comparison['match_3_significant_digits']=comparison.apply(lambda x: format(x['f1'],'.3g')==format(x['historical_f1'],'.3g') if x['status']=='rerun' else None,axis=1)
display(comparison)
comparison.to_csv(RUN/'historical_vs_rerun.csv',index=False)

## 6. 可视化与结论
下图标注历史保存指标；只有真正重跑成功才叠加本次结果。不同数据集的其他指标不放在同一个图里。

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(comparison));ax.bar(x-.15,comparison.historical_f1,.3,label='Historical stored F1')
if 'f1' in comparison:ax.bar(x+.15,comparison.f1,.3,label='Rerun F1')
ax.set_xticks(x,comparison.model,rotation=20,ha='right');ax.set_ylabel('Weighted F1')
ax.set_title('237-sample legacy evaluation: historical vs actual rerun');ax.legend()
fig.tight_layout();fig.savefig(RUN/'comparison.png',dpi=160);plt.show()
summary={'historical_metric_records':len(history),'models_rerun':int(reruns.status.eq('rerun').sum()),'models_blocked':int(reruns.status.eq('blocked').sum()),'models_failed':int(reruns.status.eq('failed').sum()),'training_rerun':False,'result_folder':str(RUN),'warning':'Historical numbers are not rerun results. Missing upstream analyses are not reproduced.'}
(RUN/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
display(summary)

## 后续恢复与原稿疑点

- 恢复 `data/prost5`、`data/sub3Di`、`data/esm_data` 后，可运行四模型推断；这不是从头训练复现。
- GO 富集 / 结构簇分析需要 `02_data/data/all`、GO 计算程序与可核验的输入；没有用现成图冒充重跑。
- `1 Function Enrichment Analysis.ipynb` cell 42 解包 `inner_dict` 却写 `value`；应核实变量后再重建该阶段。
- `3 CLEAN & SepFun.ipynb` cell 101 的各指标使用不同标签变量，暂不纳入统一比较。
- 原 notebook 有历史路径和交互状态依赖，保留不改；本 notebook 的统一路径与明确函数消除了本次主线的覆盖式重定义。
- 原 CLEAN 多标签评估只按真值标签集合编码，可能忽略集合外预测；本 notebook 为对齐历史行为保留其实现。科学复评应另列更严格口径，不能无说明替换旧值。
- 环境与每次运行状态已保存；当前 notebook 执行成功仅说明检查和已具备输入的步骤运行成功，不等于所有研究结果均已复现。

## 7. 最终 PDF 来源与 EC 层级：首标签口径 / 全标签集合口径


In [ ]:
# This source is embedded verbatim into the single reproduction notebook.
# Requires the notebook's ROOT, DATA, CLEAN, RUN, clusters and all_labels.
from scipy.stats import hypergeom
import subprocess, shutil
FINAL_PDF=Path(os.environ['THESIS_FINAL_PDF'])
(RUN/'final_pdf_provenance.json').write_text(json.dumps({'path':str(FINAL_PDF),'sha256':sha256(FINAL_PDF),'pages':77},indent=2))

# EC levels: preserve the legacy first-annotation definition and compare all-label sets.
level_rows=[]
legacy_bad=cluster_summary.loc[~cluster_summary.list_consistent,'cluster'].tolist()
for level in (1,2,3,4):
    first_count=set_count=0
    for key in legacy_bad:
        labs=[all_labels[x] for x in clusters[key]]
        first=['.'.join(x[0].split('.')[:level]) for x in labs]
        sets=[frozenset('.'.join(e.split('.')[:level]) for e in x) for x in labs]
        first_count+=len(set(first))==1
        set_count+=len(set(sets))==1
    level_rows.append(dict(level=level,denominator=len(legacy_bad),legacy_first_annotation_consistent=first_count,all_annotation_sets_consistent=set_count))
levels=pd.DataFrame(level_rows);levels.to_csv(RUN/'table4_3_ec_levels.csv',index=False);display(levels)
disagreements=cluster_summary[cluster_summary.list_consistent!=cluster_summary.set_consistent]
disagreements.to_csv(RUN/'ec_order_disagreements.csv',index=False)
non_single=cluster_summary[cluster_summary['size']>1]
print('EC consistency: all clusters',cluster_summary.list_consistent.mean(),'non-singletons',non_single.list_consistent.mean())



## 8. 蛋白宇宙映射：流式重算与覆盖问题


In [ ]:
# Streaming universe counts: row counts, not a claim of global unique protein IDs.
universe_counts=Counter();flags=Counter();total_rows=0
if availability(DATA/'Mem_all.txt')=='local':
    with (DATA/'Mem_all.txt').open() as f:
        for line in f:
            fields=line.rstrip('\n').split('\t');total_rows+=1;flags[fields[2]]+=1
            if fields[2] in {'1','2'}:universe_counts[fields[0]]+=1
    links=defaultdict(set);last_local={}
    with (DATA/'Clu_all.txt').open() as f:
        for line in f:
            u,k,*_=line.rstrip('\n').split('\t');links[u].add(k);last_local[u]=k
    mapping_rows=[]
    for u,ks in links.items():
        for k in sorted(ks):mapping_rows.append(dict(universe=u,local=k,local_size=len(clusters.get(k,[])),universe_size=universe_counts.get(u,0)))
    mapping=pd.DataFrame(mapping_rows);mapping.to_csv(RUN/'universe_mapping_recomputed.csv',index=False)
    selected=[u for u,n in universe_counts.items() if 1000<n<2000 and u in last_local and len(clusters.get(last_local[u],[]))>1]
    ureport=dict(total_rows=total_rows,flags=dict(flags),retained_member_rows=sum(universe_counts.values()),universe_clusters=len(universe_counts),mapping_local_ids=mapping.local.nunique(),mapping_global_ids=len(links),multi_global_groups=sum(len(v)>1 for v in links.values()),locals_in_multi_groups=sum(len(v) for v in links.values() if len(v)>1),single_global_groups=sum(len(v)==1 for v in links.values()),unmapped_local_clusters=sorted(set(clusters)-set(mapping.local)),subset_legacy_clusters=len(selected),subset_legacy_local_members=sum(len(clusters[last_local[u]]) for u in selected),subset_universe_member_rows=sum(universe_counts[u] for u in selected),legacy_reverse_mapping_collisions=sum(len(v)-1 for v in links.values()),mean_expansion_per_local=float((mapping.universe_size/mapping.local_size.replace(0,np.nan)).mean()))
    (RUN/'universe_summary.json').write_text(json.dumps(ureport,indent=2));display(ureport)
    cases=mapping[mapping.local.isin(['Q7N2V3','P20921'])];cases.to_csv(RUN/'table4_7_cases.csv',index=False);display(cases)



## 9. GO 富集：从原始注释重算并核对 R 历史结果


In [ ]:
# Reconstruct GO annotations directly from the dated UniProt TSV, not possibly stale HDF5 exports.
columns={'BP':'Gene Ontology (biological process)','MF':'Gene Ontology (molecular function)','CC':'Gene Ontology (cellular component)'}
eligible={k:v for k,v in clusters.items() if len(v)>10};eligible_ids={p for v in eligible.values() for p in v}
go={a:{} for a in columns}
raw_go=DATA/'uniprotkb_AND_reviewed_true_2023_07_03.tsv'
for chunk in pd.read_csv(raw_go,sep='\t',usecols=['Entry',*columns.values()],chunksize=20000,dtype=str):
    chunk=chunk[chunk.Entry.isin(eligible_ids)]
    for a,col in columns.items():
        for p,terms in zip(chunk.Entry,chunk[col]):
            if pd.notna(terms):
                ids=frozenset(re.findall(r'GO:\d{7}',terms))
                if ids:go[a][p]=ids

def bh_adjust(p):
    p=np.asarray(p,float);order=np.argsort(p);ranked=p[order]*len(p)/np.arange(1,len(p)+1)
    out=np.empty(len(p));out[order]=np.minimum(1,np.minimum.accumulate(ranked[::-1])[::-1]);return out

go_stats=[];go_cluster_rows=[];enrichments=[]
for a,annotations in go.items():
    bg=Counter(t for terms in annotations.values() for t in terms);N=len(annotations);nc=0
    for k,members in eligible.items():
        labeled=[p for p in members if p in annotations]
        if not labeled:continue
        nc+=1;ct=Counter(t for p in labeled for t in annotations[p]);terms=sorted(ct)
        # Exact one-sided over-representation; BH within each cluster over observed terms.
        pv=hypergeom.sf(np.array([ct[t] for t in terms])-1,N,np.array([bg[t] for t in terms]),len(labeled));q=bh_adjust(pv)
        for t,pval,padj in zip(terms,pv,q):enrichments.append(dict(aspect=a,cluster=k,term=t,count=ct[t],sample=len(labeled),bg_count=bg[t],background=N,pvalue=pval,p_adjust=padj))
        go_cluster_rows.append(dict(aspect=a,cluster=k,size=len(members),annotated_members=len(labeled),terms=len(terms),significant_fraction=float((q<.05).mean()),EC_consistent=bool(cluster_summary.set_index('cluster').loc[k,'list_consistent'])))
    go_stats.append(dict(aspect=a,terms=len(bg),annotated_members=N,clusters=nc))
go_stats=pd.DataFrame(go_stats);go_stats.to_csv(RUN/'table4_4_go_recomputed.csv',index=False);display(go_stats)
go_clusters=pd.DataFrame(go_cluster_rows);go_clusters.to_csv(RUN/'go_cluster_recomputed.csv',index=False)
go_enrichment=pd.DataFrame(enrichments);go_enrichment.to_csv(RUN/'go_enrichment_recomputed.csv',index=False)
go_enrichment[(go_enrichment.cluster=='Q7N2V3')&(go_enrichment.aspect=='BP')].to_csv(RUN/'table4_5_go_case_recomputed.csv',index=False)
display(go_clusters.groupby('aspect').agg(mean_terms=('terms','mean'),mean_significant_fraction=('significant_fraction','mean'),all_significant_fraction=('significant_fraction',lambda x:(x==1).mean())))
# Recover stored R results for numerical verification only; these never replace recomputed values.
if shutil.which('Rscript'):
    rcode='''args<-commandArgs(TRUE); for(a in c("BP","MF","CC")){x<-readRDS(file.path(args[1],paste0(a,"_all_data.rds"))); z<-do.call(rbind,lapply(names(x),function(k){d<-x[[k]]; d$cluster<-k; d})); z$aspect<-a; write.csv(z,file.path(args[2],paste0("stored_GO_",a,".csv")),row.names=FALSE)}'''
    subprocess.run(['Rscript','-e',rcode,str(DATA/'GO_analysis'),str(RUN)],check=True,timeout=90,capture_output=True,text=True)
    old=pd.concat([pd.read_csv(RUN/f'stored_GO_{a}.csv') for a in columns])
    joined=go_enrichment.merge(old[['aspect','cluster','ID','pvalue','p.adjust']],left_on=['aspect','cluster','term'],right_on=['aspect','cluster','ID'],how='outer',suffixes=('_rerun','_stored'),indicator=True)
    joined.to_csv(RUN/'go_stored_vs_recomputed.csv',index=False)
    go_check=dict(recomputed_rows=len(go_enrichment),stored_rows=len(old),match_status=joined['_merge'].value_counts().to_dict(),pvalues_close=int(np.isclose(joined.pvalue_rerun,joined.pvalue_stored,rtol=1e-7,atol=1e-300).sum()),adjusted_close=int(np.isclose(joined.p_adjust,joined['p.adjust'],rtol=1e-7,atol=1e-300).sum()))
    (RUN/'go_verification.json').write_text(json.dumps(go_check,indent=2));display(go_check)
fig,axs=plt.subplots(1,2,figsize=(10,4))
for a in columns:
    vals=go_clusters.loc[go_clusters.aspect==a,'significant_fraction'];axs[0].hist(vals,bins=np.linspace(0,1,11),histtype='step',label=a)
    g=go_clusters[go_clusters.aspect==a];axs[1].scatter(g['size'],g.terms,s=5,alpha=.3,label=a)
axs[0].set(xlabel='BH-adjusted p < .05 fraction',ylabel='Clusters');axs[1].set(xlabel='Cluster size',ylabel='Unique GO terms',xscale='log');axs[0].legend();axs[1].legend();fig.tight_layout();fig.savefig(RUN/'go_recomputed.png',dpi=150);plt.show()



## 10. SepFun 训练数据计数（不是模型复现）


In [ ]:
# SepFun training-label evidence (not a model rerun).
with h5py.File(DATA/'sepFun.h5','r') as f:
    sepids=list(f);folds={str(f[k][()]) for k in f}
sep_counts=dict(proteins=len(sepids),fold_ids=len(folds),EC_terms=len({e for p in sepids for e in all_labels.get(p,[])}),missing_EC_ids=sum(p not in all_labels for p in sepids))
(RUN/'sepfun_training_counts.json').write_text(json.dumps(sep_counts,indent=2));display(sep_counts)



## 11. 最终基准表的结论审计（论文报告值）


In [ ]:
# Table 4.11 manually transcribed from final PDF page 57; arithmetic audit, NOT inferred predictions.
groups=['all','unmapped','mapped','EC_consistent','EC_inconsistent']
values={'ProtT5':([.419,.502,.353,.902,.268],[.474,.483,.450,.911,.345]),'AugProtT5':([.299,.203,.327,.820,.250],[.327,.181,.404,.844,.304]),'ProstT5':([.437,.542,.370,.946,.277],[.492,.509,.475,.933,.371]),'AugProstT5':([.442,.499,.397,.924,.312],[.464,.466,.458,.911,.356])}
table=pd.DataFrame([dict(group=g,model=m,F1=f1[i],accuracy=acc[i],evidence='reported_pdf_not_model_rerun') for m,(f1,acc) in values.items() for i,g in enumerate(groups)])
table.to_csv(RUN/'table4_11_reported.csv',index=False)
deltas=[]
for base in ['ProtT5','ProstT5']:
    for i,g in enumerate(groups):deltas.append(dict(model=base,group=g,delta_F1=round(values['Aug'+base][0][i]-values[base][0][i],3),delta_accuracy=round(values['Aug'+base][1][i]-values[base][1][i],3)))
deltas=pd.DataFrame(deltas);deltas.to_csv(RUN/'table4_11_augmentation_deltas.csv',index=False);display(deltas)
fig,axs=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,base in zip(axs,['ProtT5','ProstT5']):
    d=deltas[deltas.model==base];x=np.arange(len(d));ax.barh(x-.17,d.delta_F1,.34,label='F1');ax.barh(x+.17,d.delta_accuracy,.34,label='Accuracy');ax.set_yticks(x,groups);ax.axvline(0,color='black',lw=.7);ax.set_title(base+' augmentation (reported table)');ax.legend()
fig.tight_layout();fig.savefig(RUN/'augmentation_claim_audit.png',dpi=150);plt.show()
(RUN/'final_review_run_complete.json').write_text(json.dumps({'final_pdf':str(FINAL_PDF),'model_training_rerun':False,'final_benchmark_inference_rerun':False,'GO_recomputed_from_raw_annotations':True,'universe_counts_recomputed':True},indent=2))
print('最终论文对照分析完成，输出:',RUN)


## 12. 改进分析：GO 规模分层、案例重绘与排除自相似度
本节使用本次重算表和原始注释，Mixed 案例只代表可用的一个表征通道。EC/MFO 和完整模型实验仍未复现。

In [ ]:
# Additional diagnostics. RUN is the output directory created by the notebook.
# These cells read saved, independently recomputed tables; they do not overwrite sources.
from scipy.stats import hypergeom
import json
GO_RESULTS=pd.read_csv(RUN/'go_cluster_recomputed.csv')
GO_RESULTS['size_bin']=pd.cut(GO_RESULTS['size'],[10,20,50,100,300,1000,float('inf')],right=False)
strat=GO_RESULTS.groupby(['aspect','size_bin','EC_consistent'],observed=True).agg(clusters=('cluster','size'),mean_terms=('terms','mean'),mean_significant_fraction=('significant_fraction','mean')).reset_index()
strat.to_csv(RUN/'go_size_stratified.csv',index=False)
print('按簇规模分层，供检查 EC 一致/不一致差异是否受簇大小影响；不是因果估计。');display(strat)
fig,axs=plt.subplots(2,3,figsize=(13,7))
for j,a in enumerate(['BP','MF','CC']):
    d=GO_RESULTS[GO_RESULTS.aspect==a]
    for flag,label,color in [(True,'EC consistent','gray'),(False,'EC inconsistent','salmon')]:
        v=d[d.EC_consistent==flag]
        axs[0,j].hist(v.terms,bins=30,histtype='step',color=color,label=label)
        axs[1,j].hist(v.significant_fraction,bins=np.linspace(0,1,11),histtype='step',color=color,label=label)
    axs[0,j].set(title=a,xlabel='Unique GO terms',ylabel='Clusters',yscale='log');axs[1,j].set(xlabel='Significant term fraction',ylabel='Clusters');axs[0,j].legend(fontsize=8)
fig.tight_layout();fig.savefig(RUN/'go_consistency_comparison.png',dpi=160);plt.show()

# Reconstruct the two case heatmaps directly from raw UniProt annotations.
case_members={}
with h5py.File(DATA/'Fun_fold.h5','r') as f:
    for p in f:
        v=f[p][()];v=v.decode() if isinstance(v,bytes) else str(v)
        if v in {'Q7N2V3','P20921'}:case_members.setdefault(v,[]).append(p)
ids={p for v in case_members.values() for p in v};case_go={}
for chunk in pd.read_csv(DATA/'uniprotkb_AND_reviewed_true_2023_07_03.tsv',sep='\t',usecols=['Entry','Gene Ontology (biological process)'],chunksize=30000,dtype=str):
    chunk=chunk[chunk.Entry.isin(ids)]
    for p,s in zip(chunk.Entry,chunk['Gene Ontology (biological process)']):case_go[p]=set(re.findall(r'GO:\d{7}',s)) if pd.notna(s) else set()
E=pd.read_csv(RUN/'go_enrichment_recomputed.csv')
for key,members in case_members.items():
    terms=sorted({t for p in members for t in case_go.get(p,set())});members=sorted(members)
    matrix=pd.DataFrame([[int(t in case_go.get(p,set())) for p in members] for t in terms],index=terms,columns=members)
    matrix.to_csv(RUN/f'GO_BP_{key}_presence.csv')
    fig,axs=plt.subplots(1,2,figsize=(12,5));axs[0].imshow(matrix,aspect='auto',interpolation='nearest',vmin=0,vmax=1,cmap='Greys');axs[0].set_yticks(range(len(terms)),terms,fontsize=6);axs[0].set(xlabel='Protein (IDs in CSV)',title=key+' BP annotation')
    e=E[(E.cluster==key)&(E.aspect=='BP')].sort_values('p_adjust').head(10);axs[1].barh(e.term,-np.log10(e.p_adjust.clip(lower=1e-300)));axs[1].invert_yaxis();axs[1].set(xlabel='-log10 BH-adjusted p',title='Top observed terms');fig.tight_layout();fig.savefig(RUN/f'GO_BP_{key}.png',dpi=170);plt.show()

# Available Mixed embeddings: reproduce only these panels; EC/MFO remain unavailable.
cosine_cases=[]
with h5py.File(DATA/'Enzyme_proteins_embed_dic','r') as f,h5py.File(DATA/'Miss_proteins_embed_dic','r') as extra:
    for key,members in case_members.items():
        members=sorted(members);vectors=[];missing=[]
        for p in members:
            source=f if p in f else extra
            if p not in source:missing.append(p);continue
            v=np.asarray(source[p][()]).reshape(-1);vectors.append(v)
        if missing:print('BLOCKED Mixed case',key,missing);continue
        X=np.stack(vectors).astype(float);norm=np.linalg.norm(X,axis=1)
        assert np.all(norm>0) and np.all(np.isfinite(X))
        X/=norm[:,None];S=np.clip(X@X.T,-1,1);off=S[np.triu_indices(len(S),1)]
        pd.DataFrame(S,index=members,columns=members).to_csv(RUN/f'Mixed_cosine_{key}.csv')
        cosine_cases.append(dict(cluster=key,n=len(S),legacy_all_pairs_mean=float(S.mean()),off_diagonal_mean=float(off.mean()),legacy_all_pairs_variance=float(S.var()),off_diagonal_variance=float(off.var())))
        fig,ax=plt.subplots(figsize=(5,4));im=ax.imshow(S,vmin=-1,vmax=1,cmap='viridis');fig.colorbar(im,ax=ax);ax.set(title=key+' Mixed embedding',xlabel='Protein index',ylabel='Protein index');fig.tight_layout();fig.savefig(RUN/f'Mixed_cosine_{key}.png',dpi=160);plt.show()
pd.DataFrame(cosine_cases).to_csv(RUN/'mixed_case_statistics.csv',index=False);display(pd.DataFrame(cosine_cases))
# Re-evaluate the one extreme-tail numerical discrepancy without changing the original R evidence.
z=pd.read_csv(RUN/'go_stored_vs_recomputed.csv');bad=z[~np.isclose(z.p_adjust,z['p.adjust'],rtol=1e-7,atol=1e-300)].copy()
if len(bad):
    bad['log_p_stable']=hypergeom.logsf(bad['count'].to_numpy()-1,bad.background.to_numpy(),bad.bg_count.to_numpy(),bad['sample'].to_numpy())
    bad['p_stable']=np.exp(bad.log_p_stable);bad.to_csv(RUN/'go_extreme_tail_diagnostic.csv',index=False);display(bad[['aspect','cluster','term','pvalue_rerun','pvalue_stored','p_stable']])
print('Supplement complete:',RUN)
